In [0]:
%sql
-- Databricks notebook source
-- =============================================================================
-- globalmart — Consolidated Infrastructure & Framework Setup
-- Creates: Catalog (globalmart) → Schemas (raw, bronze, silver, gold, mdm, config)
-- Creates: Unity Catalog Volume for raw files
-- Creates & Seeds: Config tables (column_mapping, dq_rules, mdm_rules)
-- Run once to bootstrap the environment.
-- =============================================================================

-- =============================================================================
-- STEP 1: Create the Catalog & Main Schemas
-- =============================================================================

CREATE CATALOG IF NOT EXISTS globalmart;
USE CATALOG globalmart;

-- RAW: Staging area for Unity Catalog Volumes
CREATE SCHEMA IF NOT EXISTS globalmart.raw;

-- BRONZE: Raw data ingested into Delta tables (no transformations)
CREATE SCHEMA IF NOT EXISTS globalmart.bronze;

-- SILVER: Harmonized, cleaned, quality-enforced tables + quarantine
CREATE SCHEMA IF NOT EXISTS globalmart.silver;

-- GOLD: Star schema for analytics
CREATE SCHEMA IF NOT EXISTS globalmart.gold;

-- MDM: Golden records
CREATE SCHEMA IF NOT EXISTS globalmart.mdm;

-- INGESTION CONTROL: Metadata tables driving dynamic pipelines
CREATE SCHEMA IF NOT EXISTS globalmart.config;

-- Source data validation table
CREATE TABLE IF NOT EXISTS globalmart.raw.schema_validation_log;

-- =============================================================================
-- STEP 2: Create the Unity Catalog Volume for raw file storage
-- =============================================================================

CREATE VOLUME IF NOT EXISTS globalmart.raw.source_files;

CREATE VOLUME IF NOT EXISTS globalmart.bronze.pipeline_metadata;
-- =============================================================================
-- STEP 3: Create Metadata Framework Tables
-- =============================================================================

-- Table 1: column_mapping
CREATE TABLE IF NOT EXISTS globalmart.config.column_mapping (
  entity              STRING      NOT NULL,
  source_column       STRING      NOT NULL,
  canonical_column    STRING      NOT NULL,
  transformation      STRING
);

-- Table 2: dq_rules
CREATE TABLE IF NOT EXISTS globalmart.config.dq_rules (
  rule_id             STRING      NOT NULL,
  entity              STRING      NOT NULL,
  rule_name           STRING      NOT NULL,
  rule_expression     STRING      NOT NULL,
  severity            STRING      NOT NULL,
  consequence         STRING      NOT NULL,
  issue_type          STRING      NOT NULL,
  failure_reason      STRING      NOT NULL
);

-- Table 3: mdm_rules (formerly survivorship_rules)
CREATE TABLE IF NOT EXISTS globalmart.config.mdm_rules (
  entity                STRING      NOT NULL,
  canonical_column      STRING      NOT NULL,
  strategy              STRING      NOT NULL,
  source_priority_order STRING
);


-- =============================================================================
-- STEP 4: Seed column_mapping
-- =============================================================================

INSERT INTO globalmart.config.column_mapping VALUES
('customers', 'customer_id',         'customer_id',    NULL),
('customers', 'customerid',          'customer_id',    NULL),
('customers', 'cust_id',             'customer_id',    NULL),
('customers', 'customer_identifier', 'customer_id',    NULL),
('customers', 'customer_email',      'customer_email', NULL),
('customers', 'email_address',       'customer_email', NULL),
('customers', 'customer_name',       'customer_name',  NULL),
('customers', 'full_name',           'customer_name',  NULL),
('customers', 'segment',             'segment',        "CASE WHEN {col} IN ('CONS', 'Cosumer') THEN 'Consumer' WHEN {col} = 'CORP' THEN 'Corporate' WHEN {col} = 'HO' THEN 'Home Office' ELSE {col} END"),
('customers', 'customer_segment',    'segment',        "CASE WHEN {col} IN ('CONS', 'Cosumer') THEN 'Consumer' WHEN {col} = 'CORP' THEN 'Corporate' WHEN {col} = 'HO' THEN 'Home Office' ELSE {col} END"),
('customers', 'country',             'country',        NULL),
('customers', 'city',                'city',           NULL),
('customers', 'state',               'state',          NULL),
('customers', 'postal_code',         'postal_code',    NULL),
('customers', 'region',              'region',         NULL);

INSERT INTO globalmart.config.column_mapping VALUES
('orders', 'order_id',                      'order_id',                           NULL),
('orders', 'customer_id',                   'customer_id',                        NULL),
('orders', 'vendor_id',                     'vendor_id',                          NULL),
('orders', 'ship_mode',                     'ship_mode',                          NULL),
('orders', 'order_status',                  'order_status',                       NULL),
('orders', 'order_purchase_date',           'order_purchase_timestamp',           "COALESCE(TRY_TO_TIMESTAMP({col}, 'MM/dd/yyyy HH:mm'), TRY_TO_TIMESTAMP({col}, 'yyyy-MM-dd HH:mm'))"),
('orders', 'order_approved_at',             'order_approved_timestamp',           "COALESCE(TRY_TO_TIMESTAMP({col}, 'MM/dd/yyyy HH:mm'), TRY_TO_TIMESTAMP({col}, 'yyyy-MM-dd HH:mm'))"),
('orders', 'order_delivered_carrier_date',  'order_delivered_carrier_timestamp',  "COALESCE(TRY_TO_TIMESTAMP({col}, 'MM/dd/yyyy HH:mm'), TRY_TO_TIMESTAMP({col}, 'yyyy-MM-dd HH:mm'))"),
('orders', 'order_delivered_customer_date', 'order_delivered_customer_timestamp', "COALESCE(TRY_TO_TIMESTAMP({col}, 'MM/dd/yyyy HH:mm'), TRY_TO_TIMESTAMP({col}, 'yyyy-MM-dd HH:mm'))"),
('orders', 'order_estimated_delivery_date', 'order_estimated_delivery_timestamp', "COALESCE(TRY_TO_TIMESTAMP({col}, 'MM/dd/yyyy HH:mm'), TRY_TO_TIMESTAMP({col}, 'yyyy-MM-dd HH:mm'))");

INSERT INTO globalmart.config.column_mapping VALUES
('transactions', 'order_id',             'order_id',             NULL),
('transactions', 'product_id',           'product_id',           NULL),
('transactions', 'sales',                'sales',                "CAST(REGEXP_REPLACE({col}, '^\\$', '') AS DECIMAL(10,2))"),
('transactions', 'quantity',             'quantity',             "CAST({col} AS INT)"),
('transactions', 'discount',             'discount',             "CAST({col} AS DECIMAL(5,2))"),
('transactions', 'profit',               'profit',               "CAST({col} AS DECIMAL(10,2))"),
('transactions', 'payment_type',         'payment_type',         "CASE WHEN {col} = '?' THEN NULL ELSE {col} END"),
('transactions', 'payment_installments', 'payment_installments', "CAST({col} AS INT)");

INSERT INTO globalmart.config.column_mapping VALUES
('returns', 'order_id',       'order_id',      NULL),
('returns', 'orderid',        'order_id',      NULL),
('returns', 'refund_amount',  'refund_amount', "CAST(REGEXP_REPLACE({col}, '^\\$', '') AS DECIMAL(10,2))"),
('returns', 'amount',         'refund_amount', "CAST({col} AS DECIMAL(10,2))"),
('returns', 'return_date',    'return_date',   "TO_DATE({col}, 'yyyy-MM-dd')"),
('returns', 'date_of_return', 'return_date',   "TO_DATE({col}, 'yyyy-MM-dd')"),
('returns', 'return_reason',  'return_reason', "CASE WHEN {col} = '?' THEN NULL ELSE {col} END"),
('returns', 'reason',         'return_reason', "CASE WHEN {col} = '?' THEN NULL ELSE {col} END"),
('returns', 'return_status',  'return_status', "CASE WHEN {col} = 'PENDG' THEN 'Pending' WHEN {col} = 'APPRVD' THEN 'Approved' WHEN {col} = 'RJCTD' THEN 'Rejected' ELSE {col} END"),
('returns', 'status',         'return_status', "CASE WHEN {col} = 'PENDG' THEN 'Pending' WHEN {col} = 'APPRVD' THEN 'Approved' WHEN {col} = 'RJCTD' THEN 'Rejected' ELSE {col} END");

INSERT INTO globalmart.config.column_mapping VALUES
('products', 'product_id',         'product_id',         NULL),
('products', 'product_name',       'product_name',       NULL),
('products', 'brand',              'brand',              NULL),
('products', 'categories',         'categories',         NULL),
('products', 'colors',             'colors',             NULL),
('products', 'manufacturer',       'manufacturer',       NULL),
('products', 'dimension',          'dimension',          NULL),
('products', 'sizes',              'sizes',              NULL),
('products', 'upc',                'upc',                NULL),
('products', 'weight',             'weight',             NULL),
('products', 'product_photos_qty', 'product_photos_qty', NULL),
('products', 'dateadded',          'date_added',         "TRY_TO_TIMESTAMP({col})"),
('products', 'dateupdated',        'date_updated',       "TRY_TO_TIMESTAMP({col})");

INSERT INTO globalmart.config.column_mapping VALUES
('vendors', 'vendor_id',   'vendor_id',   NULL),
('vendors', 'vendor_name', 'vendor_name', NULL);


-- =============================================================================
-- STEP 5: Seed dq_rules
-- =============================================================================

INSERT INTO globalmart.config.dq_rules VALUES
('CU-001', 'customers', 'customer_id_not_null',   'customer_id IS NOT NULL',                                     'CRITICAL', 'drop', 'null_primary_key',       'customer_id is null after coalescing all 4 regional ID variants'),
('CU-002', 'customers', 'valid_customer_email',   'customer_email IS NOT NULL',                                  'WARNING',  'warn', 'missing_optional_field', 'customer_email is null - Region 5 has no email column, Regions 3 and 6 have gaps'),
('CU-003', 'customers', 'valid_customer_name',    'customer_name IS NOT NULL',                                   'WARNING',  'warn', 'missing_optional_field', 'customer_name is null after coalescing customer_name and full_name'),
('CU-004', 'customers', 'valid_segment',          "segment IN ('Consumer', 'Corporate', 'Home Office')",         'WARNING',  'warn', 'invalid_category',       'segment value not in allowed set after abbreviation and typo mapping');

INSERT INTO globalmart.config.dq_rules VALUES
('OR-001', 'orders', 'order_id_not_null',         'order_id IS NOT NULL',                                        'CRITICAL', 'drop', 'null_primary_key',       'order_id is null'),
('OR-002', 'orders', 'valid_purchase_date',       'order_purchase_timestamp IS NOT NULL',                        'WARNING',  'warn', 'format_error',           'order_purchase_date could not be parsed from either MM/dd/yyyy or yyyy-MM-dd format'),
('OR-003', 'orders', 'valid_ship_mode',           'ship_mode IS NOT NULL',                                       'WARNING',  'warn', 'missing_optional_field', 'ship_mode is null - anomalously high in orders_2 (11.4%)'),
('OR-004', 'orders', 'valid_delivery_customer',   'order_delivered_customer_timestamp IS NOT NULL',              'WARNING',  'warn', 'missing_optional_field', 'order_delivered_customer_date is null - expected for undelivered orders'),
('OR-005', 'orders', 'valid_delivery_carrier',    'order_delivered_carrier_timestamp IS NOT NULL',               'WARNING',  'warn', 'missing_optional_field', 'order_delivered_carrier_date is null - expected for in-transit orders');

INSERT INTO globalmart.config.dq_rules VALUES
('TR-001', 'transactions', 'order_id_not_null',   'order_id IS NOT NULL',                                        'CRITICAL', 'drop', 'null_foreign_key',       'order_id is null - transaction cannot be linked to any order'),
('TR-002', 'transactions', 'product_id_not_null', 'product_id IS NOT NULL',                                      'CRITICAL', 'drop', 'null_foreign_key',       'product_id is null - transaction cannot be linked to any product'),
('TR-003', 'transactions', 'valid_sales',         'sales IS NOT NULL',                                           'WARNING',  'warn', 'format_error',           'sales is null after stripping $ prefix and casting to decimal'),
('TR-004', 'transactions', 'valid_profit',        'profit IS NOT NULL',                                          'WARNING',  'warn', 'missing_optional_field', 'profit is null - entire column missing from transactions_2 (Region 4)'),
('TR-005', 'transactions', 'valid_payment_type',  'payment_type IS NOT NULL',                                    'WARNING',  'warn', 'invalid_category',       'payment_type is null after replacing ? placeholders');

INSERT INTO globalmart.config.dq_rules VALUES
('RE-001', 'returns', 'order_id_not_null',        'order_id IS NOT NULL',                                        'CRITICAL', 'drop', 'null_foreign_key',       'order_id is null after coalescing order_id and orderid'),
('RE-002', 'returns', 'valid_return_status',      "return_status IN ('Pending', 'Approved', 'Rejected')",        'WARNING',  'warn', 'invalid_category',       'return_status not in allowed set after abbreviation mapping'),
('RE-003', 'returns', 'valid_return_reason',      'return_reason IS NOT NULL',                                   'WARNING',  'warn', 'missing_optional_field', 'return_reason is null after replacing ? placeholders'),
('RE-004', 'returns', 'valid_refund_amount',      'refund_amount IS NOT NULL',                                   'WARNING',  'warn', 'format_error',           'refund_amount is null after stripping $ prefix and casting to decimal');

INSERT INTO globalmart.config.dq_rules VALUES
('PR-001', 'products', 'product_id_not_null',     'product_id IS NOT NULL',                                      'CRITICAL', 'drop', 'null_primary_key',       'product_id is null'),
('PR-002', 'products', 'valid_product_name',      'product_name IS NOT NULL',                                    'WARNING',  'warn', 'missing_optional_field', 'product_name is null - product is not human-identifiable in reports'),
('PR-003', 'products', 'valid_brand',             'brand IS NOT NULL',                                           'WARNING',  'warn', 'missing_optional_field', 'brand is null - affects product performance reporting');

INSERT INTO globalmart.config.dq_rules VALUES
('VE-001', 'vendors', 'vendor_id_not_null',       'vendor_id IS NOT NULL',                                       'CRITICAL', 'drop', 'null_primary_key',       'vendor_id is null'),
('VE-002', 'vendors', 'valid_vendor_name',        'vendor_name IS NOT NULL',                                     'WARNING',  'warn', 'missing_optional_field', 'vendor_name is null - vendor is not identifiable in reports');


-- =============================================================================
-- STEP 6: Seed mdm_rules (formerly survivorship_rules)
-- =============================================================================

INSERT INTO globalmart.config.mdm_rules VALUES
('customers', 'customer_email', 'most_complete',   NULL),
('customers', 'customer_name',  'source_priority', '["customers_1","customers_2","customers_3","customers_4","customers_5","customers_6"]'),
('customers', 'segment',        'source_priority', '["customers_1","customers_2","customers_3","customers_4","customers_5","customers_6"]'),
('customers', 'city',           'most_recent',     NULL),
('customers', 'state',          'most_recent',     NULL),
('customers', 'postal_code',    'most_recent',     NULL),
('customers', 'region',         'source_priority', '["customers_1","customers_2","customers_3","customers_4","customers_5","customers_6"]'),
('customers', 'country',        'source_priority', '["customers_1","customers_2","customers_3","customers_4","customers_5","customers_6"]');




In [0]:
%sql
-- =============================================================================
-- STEP 7: Verify seed data
-- =============================================================================
SELECT entity, COUNT(*) AS mapping_count FROM globalmart.config.column_mapping GROUP BY entity ORDER BY entity;
SELECT entity, severity, COUNT(*) AS rule_count FROM globalmart.config.dq_rules GROUP BY entity, severity ORDER BY entity, severity;
SELECT * FROM globalmart.config.mdm_rules;